# Google Trends Data Collection

This section downloads and consolidates **related queries** (via API) and **related topics** (from CSV files) into one dataset.

---

## Workflow

1. **Initialize Pytrends**
   - Create a `TrendReq` object with English (US), 5-year timeframe settings.

2. **Fetch Related Queries (API)**
   - Loop through each `base_keyword`.
   - Call `pytrends.related_queries()`.
   - Extract both `top` and `rising` queries.
   - Add metadata:
     - `base_keyword` (the seed keyword)
     - `tag` (`top` or `rising`)
     - `type = query`
   - Append results to a list.

3. **Load Related Topics (CSV)**
   - For each `base_keyword`, read the corresponding `related_topics_{kw}.csv`.
   - Skip the first 3 rows (metadata).
   - Split into two parts:
     - **TOP** topics
     - **RISING** topics
   - Add metadata:
     - `base_keyword`
     - `tag` (`top` or `rising`)
     - `type = topic`
   - Append results to the same list.

4. **Combine & Save**
   - Concatenate all query and topic dataframes.
   - Save as `google_trends_all_uncleaned.csv`.

---

## Output
- **File:** `google_trends_all_uncleaned.csv`
- **Columns:**
  - `search_term` → query or topic text
  - `value` → trend score
  - `base_keyword` → seed keyword
  - `tag` → top or rising
  - `type` → query or topic


In [ ]:
# import necessary libraries
from pytrends.request import TrendReq
import pandas as pd
import time

In [ ]:
def get_google_trends_data(base_keywords, data_dir="../data/google trends", hl="en-US", tz=360, geo="US", timeframe="today 5-y"):
    """
    Fetch related queries (via Google Trends API) and related topics (from CSV),
    combine them into a single DataFrame.

    Parameters
    ----------
    base_keywords : list
        List of seed keywords to fetch related queries/topics for.
    data_dir : str, optional
        Directory where related_topics CSV files are stored. Default is "data".
    hl : str, optional
        Host language for Google Trends. Default "en-US".
    tz : int, optional
        Timezone offset. Default 360.
    geo : str, optional
        Geolocation parameter for Google Trends API. Default "US".
    timeframe : str, optional
        Timeframe parameter for Google Trends API. Default "today 5-y".

    Returns
    -------
    pd.DataFrame
        Combined dataframe of related queries and related topics.
    """

    pytrends = TrendReq(hl=hl, tz=tz)
    related_search_key = []

    # --- Related Queries (API) ---
    for kw in base_keywords:
        print(f"Processing (API queries): {kw}")
        pytrends.build_payload([kw], cat=0, geo=geo, timeframe=timeframe)
        time.sleep(1)  # Avoid rate limits

        related_queries = pytrends.related_queries()
        if kw in related_queries:
            if related_queries[kw]['top'] is not None:
                df_top_q = related_queries[kw]['top'].copy()
                df_top_q["base_keyword"] = kw
                df_top_q['tag'] = 'top'
                df_top_q['type'] = 'query'
                df_top_q.rename(columns={'query': 'search_term'}, inplace=True)
                related_search_key.append(df_top_q)

            if related_queries[kw]['rising'] is not None:
                df_rising_q = related_queries[kw]['rising'].copy()
                df_rising_q['base_keyword'] = kw
                df_rising_q['tag'] = 'rising'
                df_rising_q['type'] = 'query'
                df_rising_q.rename(columns={'query': 'search_term'}, inplace=True)
                related_search_key.append(df_rising_q)

    # --- Related Topics (CSV) ---
    for kw in base_keywords:
        print(f"Reading related topics CSV for: {kw}")
        N_ROWS_TO_SKIP = 3
        df = pd.read_csv(
            f"{data_dir}/related_topics_{kw}.csv",
            sep=",",
            skiprows=N_ROWS_TO_SKIP,
            header=None,
            names=["search_term", "value"],
            engine="python"
        )

        top_start = df[df["search_term"] == "TOP"].index[0] + 1
        rising_start = df[df["search_term"] == "RISING"].index[0] + 1

        # TOP
        df_top_t = df.iloc[top_start:rising_start-1].dropna()
        df_top_t["base_keyword"] = kw
        df_top_t["tag"] = "top"
        df_top_t["type"] = "topic"
        related_search_key.append(df_top_t)

        # RISING
        df_rising_t = df.iloc[rising_start:].dropna()
        df_rising_t["base_keyword"] = kw
        df_rising_t["tag"] = "rising"
        df_rising_t["type"] = "topic"
        related_search_key.append(df_rising_t)

    # --- Combine ---
    df_all = pd.concat(related_search_key, ignore_index=True)
    return df_all


In [ ]:
base_keywords = ["heart transplant", "liver transplant", "kidney transplant","lung transplant", "pancreas transplant"]
df = get_google_trends_data(base_keywords, data_dir="data")

df.to_csv("../data/google trends/google_trends_all_uncleaned.csv", index=False)
print("✅ Saved data to google_trends_all_uncleaned.csv")

Processing (API queries): heart transplant
Processing (API queries): liver transplant
Processing (API queries): kidney transplant
Processing (API queries): lung transplant
Processing (API queries): pancreas transplant
Reading related topics CSV for: heart transplant
Reading related topics CSV for: liver transplant
Reading related topics CSV for: kidney transplant
Reading related topics CSV for: lung transplant
Reading related topics CSV for: pancreas transplant
✅ Saved data to google_trends_all_uncleaned.csv


# Data Cleaning & Deduplication

This section processes the raw Google Trends dataset into a **clean, deduplicated, and medical-relevant dataset**.

---

## Workflow Overview

The cleaning process includes **three steps**:

1. **Exact Deduplication**  
   - Merge duplicate `search_term` entries.  
   - Combine associated `base_keywords` into a list.  

2. **Zero-Shot Classification Filtering**  
   - Use a pre-trained transformer (`facebook/bart-large-mnli`) for zero-shot classification.  
   - Keep only terms whose top predicted label is in:  
     - `["medical", "health", "disease"]`  

3. **Fuzzy Deduplication**  
   - Apply fuzzy string matching (`fuzz.ratio`) to detect and merge semantically similar terms.  
   - Ensures the final dataset is **unique, medical-relevant, and cleaned**.
   - Final dataset contains:  
   - `search_term` → main keyword kept  
   - `type` → list of associated base keywords  
   - `source` → `"google trends"`  
   - `variants` → list of merged similar terms 

In [9]:
import pandas as pd
from transformers import pipeline
from fuzzywuzzy import fuzz


In [13]:
# 1. Exact Deduplication
def exact_deduplication(df, source):
    """
    Deduplicate search terms exactly, merging base_keywords into a list.
    """
    term_to_keywords = {}
    for idx, row in df.iterrows():
        term = row['search_term']
        kw = row['base_keyword']
        if term in term_to_keywords:
            if kw not in term_to_keywords[term]:
                term_to_keywords[term].append(kw)
        else:
            term_to_keywords[term] = [kw]

    df_cleaned = pd.DataFrame([
        {"search_term": term, "type": term_to_keywords[term],"source": source}
        for term in term_to_keywords
    ])
    return df_cleaned.reset_index(drop=True)


# 2. Zero-Shot Classification Filtering
def filter_medical_terms(df, classifier=None, candidate_labels=None, blacklist=None):
    """
    Filter terms using zero-shot classification, keeping only medical-related ones.
    """
    df = df.copy()
    if candidate_labels is None:
        candidate_labels = ["medical", "health", "disease", "entertainment", "celebrity"]
    if blacklist is None:
        blacklist = ['selena', 'nick', 'bert', 'joslyn', 'nate', 'movie', 'cheney']

    if classifier is None:
            classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

    df['search_term'] = df['search_term'].astype(str).str.lower().str.strip()

    def is_medical(term):
        if not term:
            return False
        try:
            result = classifier(term, candidate_labels)
            return result['labels'][0] in ["medical", "health", "disease"]
        except Exception:
            return False
    

    df_filtered = df[df['search_term'].apply(is_medical)].reset_index(drop=True)
    # Apply blacklist filter
    if blacklist:
        import re
        esc_blacklist = [re.escape(x.lower()) for x in blacklist]
        pattern = "|".join(esc_blacklist)
        df_filtered = df_filtered[~df_filtered['search_term'].str.contains(pattern, na=False)]

    return df_filtered.reset_index(drop=True)


# 3. Fuzzy Deduplication
def fuzzy_deduplication(df, threshold=90):
    """
    Perform fuzzy deduplication based on string similarity.
    """
    groups = []  # 每个 group: {"search_term": canonical, "variants": [...], "type": [...], "source": ...}

    for _, row in df.iterrows():
        term = str(row['search_term'])
        term = term.strip()
        types = row['type'] if isinstance(row['type'], list) else [row['type']]
        source = row.get('source')

        assigned = False
        for g in groups:
            # 使用 partial_ratio 进行相似度判断
            if fuzz.partial_ratio(term, g['search_term']) >= threshold:
                g['variants'].append(term)
                g['type'].extend(types)
                assigned = True
                break

        if not assigned:
            groups.append({
                "search_term": term,
                "variants": [term],
                "type": types.copy(),
                "source": source
            })

    # 构建结果 DataFrame，去重 types 与 variants 并保留顺序
    rows = []
    for g in groups:
        # 去重并保留插入顺序
        unique_types = list(dict.fromkeys(g['type']))
        unique_variants = list(dict.fromkeys(g['variants']))
        rows.append({
            "search_term": g['search_term'],
            "type": unique_types,
            "source": g['source'],
            "variants": unique_variants
        })

    return pd.DataFrame(rows).reset_index(drop=True)

In [14]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.cluster import AgglomerativeClustering
def cluster_by_semantic_similarity(df):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(df['search_term'].tolist())

    clustering = AgglomerativeClustering(n_clusters=None, distance_threshold=1.0)
    labels = clustering.fit_predict(embeddings)

    df['cluster'] = labels

    def split_cluster_terms(terms):
        return terms[0], terms[1:]

    df_clustered = df.groupby('cluster').agg({
        'search_term': list,
        'base_keyword': lambda x: list(set(sum([k if isinstance(k, list) else [k] for k in x], [])))
    }).reset_index()

    df_clustered[['main_term', 'other_terms']] = df_clustered['search_term'].apply(
        lambda x: pd.Series(split_cluster_terms(x))
    )

    df_clustered = df_clustered.drop(columns=['search_term'])
    return df_clustered.reset_index(drop=True)

In [ ]:
# Step 1: Exact Deduplication
df_cleaned = exact_deduplication(df,source="google trends")
print("After exact deduplication:", len(df_cleaned))

# Step 2: Filter medical terms (with blacklist)
df_filtered = filter_medical_terms(df_cleaned)
print("After medical filtering:", len(df_filtered))

# Step 3: Fuzzy Deduplication
df_final = fuzzy_deduplication(df_filtered, threshold=90)
print("After fuzzy deduplication:", len(df_final))

# Save final dataset
df_final.to_csv("../data/google trends/google_trends_cleaned.csv", index=False)
print("✅ Saved to google_trends_cleaned.csv")

After exact deduplication: 265


Device set to use cuda:0


After medical filtering: 226
After fuzzy deduplication: 76
✅ Saved to google_trends_cleaned.csv
